# ShopSense — Seed Embeddings + Visual Search Test
Phase 2. Is notebook mein: (1) 7 seed products ki CLIP embeddings banti hain, (2) `seed_embeddings.json` download hota hai (backend ke `data/` folder mein rakhna hai), (3) image search aur Roman Urdu text search test hotay hain.

In [ ]:
!pip install -q transformers torch pillow numpy

In [ ]:
import time
from transformers import CLIPProcessor, CLIPModel
t0=time.time()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.eval()
print(f"CLIP loaded in {time.time()-t0:.1f}s")

## 1. Seed catalog

In [ ]:
SEED_PRODUCTS = [
 {
  "category": "footwear",
  "id": "seed-01",
  "image_file": "shoe_grey_slipon.webp",
  "platform": "daraz",
  "price_pkr": 3299,
  "purchase_link": "https://www.daraz.pk/products/grey-slipon-sample",
  "title": "Men's Grey Slip-On Casual Sneakers - Lightweight Knit",
  "title_urdu": "Mardon ke Grey Slip-On Jootay"
 },
 {
  "category": "footwear",
  "id": "seed-02",
  "image_file": "shoe_black_slipon.webp",
  "platform": "telemart",
  "price_pkr": 2899,
  "purchase_link": "https://www.telemart.pk/black-slipon-sample",
  "title": "Men's Black Slip-On Walking Shoes - White Sole",
  "title_urdu": "Mardon ke Kalay Slip-On Jootay"
 },
 {
  "category": "footwear",
  "id": "seed-03",
  "image_file": "shoe_red_nike.jpg",
  "platform": "priceoye",
  "price_pkr": 5499,
  "purchase_link": "https://www.daraz.pk/products/red-sneaker-sample",
  "title": "Red Athletic Running Sneakers - Breathable Mesh",
  "title_urdu": "Lal Running Joggers"
 },
 {
  "category": "footwear",
  "id": "seed-04",
  "image_file": "shoe_white_nike.jpg",
  "platform": "daraz",
  "price_pkr": 3999,
  "purchase_link": "https://www.daraz.pk/products/white-sneaker-sample",
  "title": "White Casual Sneakers - Everyday Comfort",
  "title_urdu": "Safaid Casual Jootay"
 },
 {
  "category": "footwear",
  "id": "seed-05",
  "image_file": "shoe_sneaker_blue.jpg",
  "platform": "telemart",
  "price_pkr": 4799,
  "purchase_link": "https://www.telemart.pk/blue-sneaker-sample",
  "title": "Blue High-Top Street Sneakers",
  "title_urdu": "Neelay Street Sneakers"
 },
 {
  "category": "watches",
  "id": "seed-06",
  "image_file": "watch_round.jpg",
  "platform": "priceoye",
  "price_pkr": 8999,
  "purchase_link": "https://www.priceoye.pk/analog-watch-sample",
  "title": "Classic Round Dial Analog Watch - Leather Strap",
  "title_urdu": "Classic Gol Ghari"
 },
 {
  "category": "bags",
  "id": "seed-07",
  "image_file": "bag_leather.jpg",
  "platform": "bagallery",
  "price_pkr": 6499,
  "purchase_link": "https://www.bagallery.com/handbag-sample",
  "title": "Women's Leather Shoulder Handbag - Tan",
  "title_urdu": "Khwateen ka Chamray ka Handbag"
 }
]
print(f"{len(SEED_PRODUCTS)} seed products")

## 2. Seed images upload karein
Neeche wali 7 files upload karein: `shoe_grey_slipon.webp`, `shoe_black_slipon.webp`, `shoe_red_nike.jpg`, `shoe_white_nike.jpg`, `shoe_sneaker_blue.jpg`, `watch_round.jpg`, `bag_leather.jpg`

In [ ]:
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()
print(f"{len(uploaded)} files uploaded: {sorted(uploaded.keys())}")

## 3. Embeddings banayein

In [ ]:
import torch, numpy as np

def embed(image):
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
    feats = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    return feats[0].detach().cpu().numpy()

embeddings = {}
for p in SEED_PRODUCTS:
    fn = p["image_file"]
    assert fn in uploaded, f"MISSING: {fn} upload nahi hui!"
    img = Image.open(io.BytesIO(uploaded[fn])).convert("RGB")
    t0 = time.time()
    vec = embed(img)
    embeddings[p["id"]] = vec.tolist()
    print(f"{p['id']}: dim={vec.shape[0]}  {1000*(time.time()-t0):.0f}ms  norm={np.linalg.norm(vec):.4f}")
print(f"\nTotal {len(embeddings)} embeddings ready.")

## 4. `seed_embeddings.json` download karein
Ye file backend ke `data/` folder mein rakhein — phir `/search` endpoint kaam karega.

In [ ]:
with open("seed_embeddings.json", "w") as f:
    json.dump(embeddings, f)
files.download("seed_embeddings.json")
print("downloaded — isay shopsense-backend/data/seed_embeddings.json mein rakhein")

## 5. Visual search test
Koi query image upload karein (misal: kala joota) — catalog rank hokar aayega.

In [ ]:
q_up = files.upload()
qn = list(q_up.keys())[0]
qimg = Image.open(io.BytesIO(q_up[qn])).convert("RGB")
qv = embed(qimg)

def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b))

ranked = sorted(
    ((cos(qv, np.array(embeddings[p["id"]])), p) for p in SEED_PRODUCTS),
    key=lambda x: x[0], reverse=True,
)
print(f"Query: {qn}\n")
for score, p in ranked:
    print(f"{score:.4f} | PKR {p['price_pkr']} | {p['platform']} | {p['title'][:55]}")

## 6. Roman Urdu text search test

In [ ]:
RU_MAP = {
 "bag": [
  "handbag",
  "bag",
  "bags"
 ],
 "earbuds": [
  "earbuds",
  "audio",
  "wireless"
 ],
 "ghari": [
  "watch",
  "watches"
 ],
 "handbag": [
  "handbag",
  "bag",
  "bags"
 ],
 "jeans": [
  "jeans",
  "denim",
  "pants"
 ],
 "joggers": [
  "shoes",
  "sneakers",
  "running"
 ],
 "joota": [
  "shoes",
  "sneakers",
  "footwear"
 ],
 "jootay": [
  "shoes",
  "sneakers",
  "footwear"
 ],
 "kala joota": [
  "black",
  "shoes",
  "sneakers",
  "footwear"
 ],
 "kalay jootay": [
  "black",
  "shoes",
  "sneakers",
  "footwear"
 ],
 "kurta": [
  "kurta",
  "ethnic",
  "dress"
 ],
 "ladies bag": [
  "handbag",
  "bag",
  "bags"
 ],
 "lal joota": [
  "red",
  "shoes",
  "sneakers",
  "footwear"
 ],
 "lawn kurta": [
  "kurta",
  "lawn",
  "ethnic"
 ],
 "safaid joota": [
  "white",
  "shoes",
  "sneakers",
  "footwear"
 ],
 "smartwatch": [
  "watch",
  "smartwatch"
 ],
 "sneaker": [
  "shoes",
  "sneakers",
  "footwear"
 ],
 "suit": [
  "suit",
  "kurta",
  "ethnic"
 ],
 "watch": [
  "watch",
  "watches"
 ]
}

def normalize_query(query):
    q = query.strip().lower()
    keywords = []
    for phrase in sorted(RU_MAP, key=len, reverse=True):
        if phrase in q:
            for kw in RU_MAP[phrase]:
                if kw not in keywords:
                    keywords.append(kw)
            q = q.replace(phrase, " ")
    for w in q.split():
        if w and w not in keywords:
            keywords.append(w)
    return keywords

for test in ["kala joota", "smartwatch", "ladies bag"]:
    kws = normalize_query(test)
    hits = []
    for p in SEED_PRODUCTS:
        hay = f"{p['title']} {p.get('title_urdu','')} {p.get('category','')}".lower()
        s = sum(1 for kw in kws if kw in hay)
        if s: hits.append((s, p))
    hits.sort(key=lambda x: x[0], reverse=True)
    print(f"'{test}' -> {kws}")
    for s, p in hits[:3]:
        print(f"   {s}/{len(kws)} | PKR {p['price_pkr']} | {p['title'][:50]}")
    print()